In [11]:
QUERY_AUDIO_PATH = "C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav"

FILENAME= QUERY_AUDIO_PATH.split("/")[-1].split(".")[0]
FILENAME

'dark-distorted-hard-trap-bass_F_minor'

In [12]:
import asyncio
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from qdrant_client import QdrantClient
import sys
import os

# Setup Path
sys.path.append(os.path.abspath("../.."))
from config.settings import settings
from rag.clap.model_handler import create_clap_model

# --- CONFIGURAZIONE ---
QDRANT_COLLECTION = settings.QDRANT_ENRICHED_COLLECTION_NAME
BACKGROUND_SAMPLES = 400       # Punti di contesto (rumore di fondo)
TOP_K = 10                     # Quanti risultati recuperare

async def visualize_audio_semantic_map():
    if not os.path.exists(QUERY_AUDIO_PATH):
        print(f"ERRORE: Il file '{QUERY_AUDIO_PATH}' non è stato trovato nella cartella corrente.")
        return

    print(f"--- ANALISI RETRIEVAL AUDIO: '{QUERY_AUDIO_PATH}' ---")

    # Init Clients
    client_q = QdrantClient(host=settings.QDRANT_CONNECTION_HOST, port=settings.QDRANT_PORT)

    print("Caricamento modello CLAP...")
    clap = create_clap_model()

    # 1. EMBEDDING DELLA QUERY (Audio Input)
    # Usiamo CLAP per convertire il file WAV in un vettore
    print("Calcolo embedding audio input...")
    query_vector = clap.get_audio_embedding([QUERY_AUDIO_PATH])[0]

    # 2. RETRIEVAL (I Vicini più prossimi)
    # Cerchiamo nello spazio 'audio_vector' (Similarità Acustica)
    print("Esecuzione ricerca vettoriale (Audio-to-Audio)...")
    search_res = client_q.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        using="audio_vector", # Fondamentale: usiamo lo spazio audio
        limit=TOP_K,
        with_payload=True,
        with_vectors=["audio_vector"] # Ci serve il vettore per plottarlo!
    )

    # 3. BACKGROUND (Il Contesto)
    # Prendiamo campioni casuali dallo spazio audio per disegnare la "mappa"
    print("Recupero contesto di fondo...")
    scroll_res, _ = client_q.scroll(
        collection_name=QDRANT_COLLECTION,
        limit=BACKGROUND_SAMPLES,
        with_payload=True,
        with_vectors=["audio_vector"]
    )

    # --- PREPARAZIONE DATI (Stessa struttura dello script testuale) ---
    vectors = []
    meta = []

    # A. Aggiungiamo la Query (Indice 0)
    vectors.append(query_vector)
    meta.append({
        "name": "INPUT AUDIO",
        "desc": f"File: {QUERY_AUDIO_PATH}",
        "score": 1.0,
        "type": "query"
    })

    # B. Aggiungiamo i Risultati (Indici 1...K)
    result_indices = []
    for i, hit in enumerate(search_res.points):
        # Gestione sicura del vettore (dict o list)
        vec = hit.vector.get("audio_vector") if isinstance(hit.vector, dict) else hit.vector
        if vec is None: continue

        vectors.append(vec)
        payload = hit.payload or {}

        # Costruiamo una descrizione leggibile
        fname = payload.get("original_filename", "Unknown")
        label = payload.get("label", "")
        tags = payload.get("ai_tags", [])
        desc_text = f"Label: {label}<br>Tags: {', '.join(tags[:3])}"

        meta.append({
            "name": fname,
            "desc": desc_text,
            "score": hit.score,
            "type": "result"
        })
        result_indices.append(len(vectors) - 1)

    # C. Aggiungiamo il Background
    for hit in scroll_res:
        # Evitiamo duplicati se il background include i risultati trovati
        if hit.id in [r.id for r in search_res.points]:
            continue

        vec = hit.vector.get("audio_vector") if isinstance(hit.vector, dict) else hit.vector
        if vec is None: continue

        vectors.append(vec)
        payload = hit.payload or {}
        meta.append({
            "name": "Background",
            "desc": payload.get("label", "No Label"),
            "score": 0,
            "type": "noise"
        })

    # --- RIDUZIONE DIMENSIONALE ---
    print("Proiezione 2D (PCA + t-SNE)...")
    X = np.array(vectors)

    # Step 1: PCA
    pca = PCA(n_components=min(50, len(X)))
    X_pca = pca.fit_transform(X)

    # Step 2: t-SNE
    tsne = TSNE(n_components=2, perplexity=min(30, len(X)-1), random_state=42, init='pca', learning_rate='auto')
    X_2d = tsne.fit_transform(X_pca)

    # --- VISUALIZZAZIONE INTERATTIVA (Stesso codice grafico) ---
    fig = go.Figure()

    # 1. Disegna le CONNESSIONI (Query -> Risultati)
    q_x, q_y = X_2d[0] # Coordinate Query

    for idx in result_indices:
        r_x, r_y = X_2d[idx]
        score = meta[idx]['score']

        fig.add_trace(go.Scatter(
            x=[q_x, r_x], y=[q_y, r_y],
            mode='lines',
            line=dict(
                color='rgba(0, 0, 255, 0.4)', # Blu per l'audio (per distinguerlo dal rosso del testo)
                width=score * 5
            ),
            hoverinfo='none',
            showlegend=False
        ))

    # 2. Disegna il BACKGROUND (Grigio)
    bg_indices = [i for i, m in enumerate(meta) if m['type'] == 'noise']
    bg_x = X_2d[bg_indices, 0]
    bg_y = X_2d[bg_indices, 1]
    bg_text = [meta[i]['desc'] for i in bg_indices]

    fig.add_trace(go.Scatter(
        x=bg_x, y=bg_y,
        mode='markers',
        marker=dict(size=6, color='#bdc3c7', opacity=0.4),
        text=bg_text,
        name='Context (Database)',
        hoverinfo='text'
    ))

    # 3. Disegna i RISULTATI (Colorati per Score)
    res_x = [X_2d[i][0] for i in result_indices]
    res_y = [X_2d[i][1] for i in result_indices]
    res_scores = [meta[i]['score'] for i in result_indices]
    res_text = [f"<b>{meta[i]['name']}</b><br>Score: {meta[i]['score']:.3f}<br>{meta[i]['desc']}" for i in result_indices]

    fig.add_trace(go.Scatter(
        x=res_x, y=res_y,
        mode='markers',
        marker=dict(
            size=15,
            color=res_scores,
            colorscale='Blues', # Scala Blu per Audio
            showscale=True,
            colorbar=dict(title="Similarity Score"),
            line=dict(color='white', width=2)
        ),
        text=res_text,
        name='Similar Sounds',
        hoverinfo='text'
    ))

    # 4. Disegna la QUERY (Stella Rossa)
    fig.add_trace(go.Scatter(
        x=[q_x], y=[q_y],
        mode='markers',
        marker=dict(size=25, color='red', symbol='star', line=dict(color='black', width=1)),
        name='Input Audio File',
        hoverinfo='name'
    ))

    # Layout
    fig.update_layout(
        title=f"<b>Audio Retrieval Map</b><br>Input: {FILENAME}",
        width=1000,
        height=800,
        template="plotly_white",
        xaxis=dict(showgrid=False, zeroline=False, visible=False),
        yaxis=dict(showgrid=False, zeroline=False, visible=False),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )

    fig.show()

# Esecuzione
# asyncio.run(visualize_audio_semantic_map())
await visualize_audio_semantic_map()

--- ANALISI RETRIEVAL AUDIO: 'C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav' ---
Caricamento modello CLAP...
Calcolo embedding audio input...
Esecuzione ricerca vettoriale (Audio-to-Audio)...
Recupero contesto di fondo...
Proiezione 2D (PCA + t-SNE)...


In [13]:
import asyncio
import os
import pandas as pd
from qdrant_client import QdrantClient
import sys

# Setup Path
sys.path.append(os.path.abspath(".."))
from config.settings import settings
from rag.clap.model_handler import create_clap_model

# --- CONFIGURAZIONE ---
QDRANT_COLLECTION = settings.QDRANT_ENRICHED_COLLECTION_NAME
TOP_K = 10                     # Quanti risultati vedere

async def print_audio_retrieval_table():
    if not os.path.exists(QUERY_AUDIO_PATH):
        print(f"❌ ERRORE: File '{QUERY_AUDIO_PATH}' non trovato.")
        return

    print(f"--- TABELLA RISULTATI AUDIO-TO-AUDIO: '{QUERY_AUDIO_PATH}' ---")

    # 1. Init
    client_q = QdrantClient(host=settings.QDRANT_CONNECTION_HOST, port=settings.QDRANT_PORT)
    clap = create_clap_model()

    # 2. Embedding Audio Input
    print("Calcolo embedding audio...")
    query_vector = clap.get_audio_embedding([QUERY_AUDIO_PATH])[0]

    # 3. Ricerca
    print("Esecuzione ricerca...")
    search_res = client_q.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        using="audio_vector", # Ricerca per similarità acustica
        limit=TOP_K,
        with_payload=True
    )

    # 4. Costruzione Tabella
    table_data = []

    for rank, hit in enumerate(search_res.points, start=1):
        payload = hit.payload or {}

        # Estrazione Dati
        filename = payload.get("original_filename", "Unknown")

        # Gestione Label: AI Label (Arricchita) vs Original Label (Grezza)
        ai_label = payload.get("ai_label", "N/A")
        orig_label = payload.get("original_label") or payload.get("original_filename", "N/A")

        # Tags (prendiamo i primi 3 per brevità)
        tags = payload.get("ai_tags", [])
        tags_str = ", ".join(tags[:5]) if isinstance(tags, list) else str(tags)

        # CLAP Quality Score (memorizzato nel DB durante l'ingestion)
        # Questo score indica quanto la descrizione AI è fedele all'audio di quel file
        stored_quality = payload.get("clap_score", 0.0)

        table_data.append({
            "Rank": rank,
            "Similarity Score": hit.score, # Quanto è simile al TUO input file
            #"Filename": filename,
            "AI Description": ai_label,
            "Original Label": orig_label,
            "Tags": tags_str,
            "Label Quality": stored_quality
        })

    # Creazione DataFrame
    df = pd.DataFrame(table_data)

    # --- FORMATTAZIONE VISIVA ---
    print("\n" + "="*80)
    print(f"TOP-{TOP_K} FILE ACUSTICAMENTE SIMILI A: {QUERY_AUDIO_PATH}")
    print("="*80)

    # Opzioni Pandas per vedere tutto il testo
    pd.set_option('display.max_colwidth', None)

    # Stile per evidenziare i punteggi alti
    # - Similarity Score: Colore Blu (Similarità acustica)
    # - Label Quality: Colore Verde (Qualità della descrizione nel DB)
    display(df.style.background_gradient(subset=['Similarity Score'], cmap='Blues')
                    .background_gradient(subset=['Label Quality'], cmap='Greens')
                    .format({'Similarity Score': '{:.4f}', 'Label Quality': '{:.4f}'}))

# Esecuzione
# asyncio.run(print_audio_retrieval_table())
await print_audio_retrieval_table()

--- TABELLA RISULTATI AUDIO-TO-AUDIO: 'C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav' ---
Calcolo embedding audio...
Esecuzione ricerca...

TOP-10 FILE ACUSTICAMENTE SIMILI A: C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav


,Rank,Similarity Score,AI Description,Original Label,Tags,Label Quality
0,1,0.6286,Bass-line with aggressive and dynamic slides,UK Drill 808 with Smooth Slides,"aggressive, dynamic, heavy, energetic",0.1598
1,2,0.6254,Bass-line with Buzzy Warm Texture and Dry Clean Tone,808 Buzzy Bass One-Shot,"warm, dry, clean, energetic",0.4003
2,3,0.6134,"Bass-line with Clean, Dark, and Energetic Character",Dubstep Bass Line - Hard,"dynamic, clean, dark, dry, energetic",0.3183
3,4,0.6115,Digital Synth Bass with Aggressive and Dry Character,Danger Reese Bass,"aggressive, clean, dry, heavy",0.4893
4,5,0.6052,"Sub-bass with flowing and heavy timbre, short one-shot articulation",Spring 808 Bass,"short, one shot, flowing, heavy",0.4029
5,6,0.6026,"Analog Synth Bass with Hard, Dark, and Dry Texture",Phonk Bass Shot - Midnight Hit,"dynamic, hard, clean, analog, dark",0.2842
6,7,0.5913,"Sub-bass with heavy, clean, and dry character",Heavy Drill 808,"dynamic, clean, heavy, dry, energetic",0.2521
7,8,0.5840,"Bass-line with heavy, boomy stabs and compressed, dark texture",Bass Line Type Kick Loop - Bouncy Stabs,"staccato, stabs, low, pulsating, heavy",0.2763
8,9,0.5825,Sub-bass with Aggressive Warm Tone and Dry Texture,Aggressive Bass Hit,"aggressive, warm, clean, dry, angry",0.3129
9,10,0.5767,Analog Synth Bass with Glitchy Pulse Texture,Phonk Bass Shot - Glitch Pulse,"dark, hip hop, chiptune, loop",0.1603
